In [1]:
!pip install kaggle -q

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"fatsedz","key":"654e24d8482a4b00f71d965031995d5f"}'}

In [3]:
import os
os.makedirs("/root/.kaggle", exist_ok=True)
os.replace("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)
print("kaggle.json set up.")


kaggle.json set up.


In [4]:
!kaggle datasets download nltkdata/brown-corpus -p /content -q --unzip

Dataset URL: https://www.kaggle.com/datasets/nltkdata/brown-corpus
License(s): other


In [5]:
import os

In [6]:
print("📁The brown folder contains the following files:")
print(os.listdir("/content/brown")[:20])

📁The brown folder contains the following files:
['brown']


In [10]:
import pandas as pd

folder = "/content/brown/brown"

texts = []
categories = []
for filename in os.listdir(folder):
    path = os.path.join(folder, filename)
    with open(path, "r", encoding="latin-1") as f:
        text = f.read()
    texts.append(text)
    categories.append(filename[:2])

df = pd.DataFrame({"category": categories, "text": texts})
print("All rows:", df.shape)
df.head()

df = df[df["category"].isin(["cf", "cs"])].copy()
print("After cf/cs filter:", df["category"].value_counts())

df.loc[df["category"] == "cf", "topic"] = "humor"
df.loc[df["category"] == "cs", "topic"] = "science_fiction"

df[["category", "topic", "text"]].head()

All rows: (503, 2)
After cf/cs filter: category
cf    48
Name: count, dtype: int64


,category,topic,text
16,cf,humor,The/at missionary/nn obligation/nn to/to procl...
25,cf,humor,Everyone/pn with/in a/at personal/jj or/cc gro...
36,cf,humor,\n\n\tIn/in Ireland's/np$ County/nn-tl Limeric...
38,cf,humor,A/at little/ql farther/rbr along/in the/at roa...
42,cf,humor,\n\n\tSome/dti recent/jj writings/nns assume/v...


In [11]:
!python -m spacy download en_core_web_sm -q

import spacy
nlp = spacy.load("en_core_web_sm")

def lemmatize(text):
    doc = nlp(str(text))
    tokens = [
        t.lemma_
        for t in doc
        if not (t.is_stop or t.is_punct or t.is_space)
    ]
    return " ".join(tokens)

print("⏳ Processing texts (may take a while)...")
df["processed_text"] = df["text"].apply(lemmatize)
df[["topic", "processed_text"]].head()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
⏳ Processing texts (may take a while)...


,topic,processed_text
16,humor,missionary nn obligation nn proclaim vb gospel...
25,humor,pn personal jj cc group nn tragedy nn relate v...
36,humor,Ireland's np$ County nn tl Limerick np tl near...
38,humor,little ql far rbr road nn ppss come vb Church ...
42,humor,dti recent jj writing nn assume vb cs ignorant...


In [12]:
from sklearn.model_selection import train_test_split

# map  0, 1
df.loc[df["topic"] == "humor", "label"] = 0
df.loc[df["topic"] == "science_fiction", "label"] = 1
df["label"] = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df["processed_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

Train size: 38
Test size : 10


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)    # unigram + bigram
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)

print("✅ Accuracy:", accuracy_score(y_test, y_pred))

unique = sorted(np.unique(y_test))
print("Unique labels in y_test:", unique)

name_map = {0: "humor", 1: "science_fiction"}
target_names = [name_map[u] for u in unique]

print("\n📊 Classification report:\n")
print(classification_report(
    y_test,
    y_pred,
    labels=unique,
    target_names=target_names,
    zero_division=0
))

✅ Accuracy: 1.0
Unique labels in y_test: [np.int64(0)]

📊 Classification report:

              precision    recall  f1-score   support

       humor       1.00      1.00      1.00        10

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



In [15]:
out_path = "/content/brown_humor_scifi_processed.csv"
df.to_csv(out_path, index=False)
print("📁 فایل ذخیره شد در:", out_path)


📁 فایل ذخیره شد در: /content/brown_humor_scifi_processed.csv
